## Sistema de Informações Hospitalares do Sistema Único de Saúde - SIH/SUS

SIH/SUS Bronze -> Silver. Executar com %run tratar_sih_silver.py no Jupyter.
Dependencias: pandas, pyarrow. Nao acessa PostgreSQL nem altera a Bronze.
Retomada: reutiliza somente particoes verificadas da MESMA execucao.
Para iniciar outra execucao, altere EXECUCAO. Nao execute duas instancias juntas.

In [3]:
from pathlib import Path
from datetime import datetime, date, timezone
from decimal import Decimal, InvalidOperation
from collections import Counter
import hashlib
import json
import re
import traceback
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

In [4]:
PROJETO = Path(r'D:\Cleber\GitHub\Projetos\observatorio_covid19_brasil')
BRONZE = PROJETO / 'Camada_Bronze/dados_sih_sus/parquet'
IBGE = PROJETO / 'Camada_Silver/dados/populacao_ibge/20261024_163052_e1db3d2b/populacao_municipios_2020_2024.parquet'
EXECUCAO = 'sih_2020_2024_v1'
LOTE = 50_000
VERSAO = '1.0.0'
UFS = set('AC AL AP AM BA CE DF ES GO MA MT MS MG PA PB PR PE PI RJ RN RS RO RR SC SP SE TO'.split())
ESPERADO = {2020: 10688203, 2021: 11629005, 2022: 12520914, 2023: 13355814, 2024: 14171364}
BASE = PROJETO / 'Camada_Silver'
DADOS = BASE / 'dados/sih' / EXECUCAO
QUALIDADE = BASE / 'qualidade/sih' / EXECUCAO
DOC = BASE / 'documentacao/sih' / EXECUCAO
DATAS = {'DT_INTER': 'data_internacao', 'DT_SAIDA': 'data_saida', 'NASC': 'data_nascimento'}
QUANTIDADES = ['IDADE', 'DIAS_PERM', 'QT_DIARIAS', 'UTI_MES_IN', 'UTI_MES_AN', 'UTI_MES_AL', 'UTI_MES_TO', 'UTI_INT_IN', 'UTI_INT_AN', 'UTI_INT_AL', 'UTI_INT_TO', 'DIAR_ACOM']
OBRIGATORIAS = set(DATAS) | set(QUANTIDADES) | set('N_AIH CNES IDENT MORTE SEXO COD_IDADE MUNIC_RES MUNIC_MOV ANO_CMPT MES_CMPT _UF_ARQUIVO _COMPETENCIA_ARQUIVO _DBF_EXCLUIDO VAL_TOT VAL_SH VAL_SP'.split())

In [6]:
def salvar_json(caminho, obj):
    temporario = caminho.with_suffix('.json.tmp')
    temporario.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding='utf-8')
    temporario.replace(caminho)


def sha256(p):
    h = hashlib.sha256()
    with p.open('rb') as f:
        for bloco in iter(lambda: f.read(8 * 1024 * 1024), b''):
            h.update(bloco)
    return h.hexdigest()


def texto(s):
    return s.astype('string').str.strip().replace('', pd.NA)


def datas(s):
    # Valida valores unicos antes de criar qualquer timestamp pandas.
    v = texto(s)
    mapa = {}
    for valor in v.dropna().unique():
        try:
            if not re.fullmatch(r'[0-9]{8}', valor):
                raise ValueError()
            d = datetime.strptime(valor, '%Y%m%d').date()
            if not date(1800, 1, 1) <= d <= date(2100, 12, 31):
                raise ValueError()
            mapa[valor] = d
        except ValueError:
            mapa[valor] = None
    resultado = v.map(mapa).astype(object).where(v.notna(), None)
    resultado = resultado.where(pd.notna(resultado), None)
    status = pd.Series('VALIDA', index=s.index, dtype='string')
    status.loc[resultado.isna()] = 'INVALIDA_OU_FORA_LIMITE_1800_2100'
    status.loc[v.isna()] = 'AUSENTE'
    return resultado, status


def numero(s, monetario):
    v = texto(s)
    def converter(x):
        if pd.isna(x):
            return None
        padrao = r'[+-]?[0-9]+(?:\.[0-9]{1,2})?' if monetario else r'[+-]?[0-9]+'
        if not re.fullmatch(padrao, x):
            return None
        try:
            n = Decimal(x) if monetario else int(x)
            if abs(n) >= (Decimal('10000000000000000') if monetario else 2**63):
                return None
            return n.quantize(Decimal('0.01')) if monetario else n
        except (InvalidOperation, ValueError):
            return None
    # Cache evita parse repetido; preserva valores invalidos na coluna original.
    cache = {x: converter(x) for x in v.dropna().unique()}
    out = pd.Series([cache.get(x) if pd.notna(x) else None for x in v], index=s.index, dtype=object)
    status = pd.Series('VALIDO', index=s.index, dtype='string')
    status.loc[out.isna()] = 'INVALIDO'
    status.loc[v.isna()] = 'AUSENTE'
    return out, status


def referencia():
    if not IBGE.is_file():
        raise FileNotFoundError(IBGE)
    ref = pd.read_parquet(IBGE)
    exigidas = {'ano_analise', 'codigo_ibge_7', 'municipio', 'uf', 'populacao'}
    if not exigidas <= set(ref):
        raise ValueError(f'IBGE: faltam {exigidas - set(ref)}')
    ref['ano_analise'] = pd.to_numeric(ref['ano_analise'], errors='raise').astype(int)
    ref['codigo_ibge_7'] = texto(ref['codigo_ibge_7'])
    if not ref['codigo_ibge_7'].str.fullmatch(r'[0-9]{7}', na=False).all():
        raise ValueError('IBGE possui codigo de 7 digitos invalido.')
    ref['codigo6'] = ref['codigo_ibge_7'].str[:6]
    if ref.duplicated(['ano_analise', 'codigo6']).any():
        raise ValueError('Referencia IBGE ambigua por ano/codigo de 6 digitos.')
    return {ano: grupo.set_index('codigo6') for ano, grupo in ref.groupby('ano_analise')}


def esquema(colunas):
    campos = [pa.field(c.lower(), pa.string()) for c in colunas]
    for nome in ['arquivo_origem', 'uf_processamento', 'competencia_processamento', 'tratado_em_utc']:
        campos.append(pa.field(nome, pa.string()))
    for nome in ['linha_arquivo', 'ano_processamento', 'mes_processamento', 'ano_internacao', 'ano_saida', 'ano_referencia_vinculo']:
        campos.append(pa.field(nome, pa.int64()))
    for nome in DATAS.values():
        campos += [pa.field(nome, pa.date32()), pa.field('status_' + nome, pa.string())]
    for c in colunas:
        if c.startswith('VAL_') or c in QUANTIDADES:
            campos += [pa.field(c.lower() + '_numerico', pa.decimal128(18, 2) if c.startswith('VAL_') else pa.int64()), pa.field('status_' + c.lower(), pa.string())]
    for papel in ['residencia', 'ocorrencia']:
        for nome in ['codigo_ibge_7_', 'municipio_ibge_', 'uf_ibge_', 'status_vinculo_']:
            campos.append(pa.field(nome + papel, pa.string()))
    for nome in ['fl_saida_antes_internacao', 'fl_nascimento_apos_internacao', 'fl_internacao_fora_2020_2024', 'fl_dois_anos_internacao_processamento_diferentes']:
        campos.append(pa.field(nome, pa.bool_()))
    if len(set(f.name for f in campos)) != len(campos):
        raise ValueError('Nomes de colunas colidem apos normalizacao.')
    return pa.schema(campos)


def tratar(df, ano, mes, uf, nome, inicio, ref, rel, pendencias, instante):
    out = df.copy()
    out.columns = out.columns.str.lower()
    n = len(df)
    for c, esperado in [('ANO_CMPT', str(ano)), ('MES_CMPT', str(mes).zfill(2)), ('_UF_ARQUIVO', uf), ('_COMPETENCIA_ARQUIVO', f'{ano}{mes:02d}')]:
        v = texto(df[c])
        if c in ['ANO_CMPT', 'MES_CMPT']:
            ok = pd.to_numeric(v, errors='coerce').eq(int(esperado))
        else:
            ok = v.eq(esperado)
        if not ok.fillna(False).all():
            raise ValueError(f'Metadado divergente: {nome}/{c}')
    out['arquivo_origem'] = nome
    out['linha_arquivo'] = range(inicio + 1, inicio + n + 1)
    out['uf_processamento'] = uf
    out['competencia_processamento'] = f'{ano}{mes:02d}'
    out['ano_processamento'] = ano
    out['mes_processamento'] = mes
    out['ano_referencia_vinculo'] = ano
    out['tratado_em_utc'] = instante
    for c, destino in DATAS.items():
        out[destino], out['status_' + destino] = datas(df[c])
        rel.update({'datas/' + destino + '/' + str(k): int(v) for k, v in out['status_' + destino].value_counts().items()})
    for c in df:
        if c.startswith('VAL_') or c in QUANTIDADES:
            v, status = numero(df[c], c.startswith('VAL_'))
            out[c.lower() + '_numerico'] = v
            out['status_' + c.lower()] = status
            rel.update({'numeros/' + c + '/' + str(k): int(q) for k, q in status.value_counts().items()})
    for campo in ['internacao', 'saida']:
        out['ano_' + campo] = pd.Series([d.year if d else None for d in out['data_' + campo]], dtype='Int64')
    a, b, c = out['data_internacao'], out['data_saida'], out['data_nascimento']
    out['fl_saida_antes_internacao'] = [y < x if x and y else False for x, y in zip(a, b)]
    out['fl_nascimento_apos_internacao'] = [y > x if x and y else False for x, y in zip(a, c)]
    out['fl_internacao_fora_2020_2024'] = out['ano_internacao'].notna() & ~out['ano_internacao'].between(2020, 2024)
    out['fl_dois_anos_internacao_processamento_diferentes'] = (out['ano_internacao'].notna() & out['ano_internacao'].ne(ano)).fillna(False)
    for coluna in [x for x in out if x.startswith('fl_')]:
        rel['avisos/' + coluna] += int(out[coluna].sum())
    for campo in ['IDENT', 'MORTE', 'SEXO', 'COD_IDADE', '_DBF_EXCLUIDO']:
        rel.update({'categorias/' + campo + '/' + str(k): int(v) for k, v in texto(df[campo]).fillna('<AUSENTE>').value_counts().items()})
    for campo, tamanho in [('N_AIH', 13), ('CNES', 7), ('MUNIC_RES', 6), ('MUNIC_MOV', 6)]:
        rel['formatos_invalidos/' + campo] += int((~texto(df[campo]).str.fullmatch(r'[0-9]{' + str(tamanho) + '}', na=False)).sum())
    r = ref[ano]
    for papel, campo in [('residencia', 'MUNIC_RES'), ('ocorrencia', 'MUNIC_MOV')]:
        codigo = texto(df[campo])
        destino = codigo.map(r['codigo_ibge_7'])
        status = pd.Series('SEM_CORRESPONDENCIA', index=df.index, dtype='string')
        status.loc[destino.notna()] = 'VINCULADO_COM_POPULACAO'
        status.loc[destino.notna() & codigo.map(r['populacao']).isna()] = 'VINCULADO_SEM_POPULACAO'
        status.loc[~codigo.str.fullmatch(r'[0-9]{6}', na=False)] = 'FORMATO_INVALIDO'
        status.loc[codigo.isna()] = 'CODIGO_AUSENTE'
        out['codigo_ibge_7_' + papel] = destino
        out['municipio_ibge_' + papel] = codigo.map(r['municipio'])
        out['uf_ibge_' + papel] = codigo.map(r['uf'])
        out['status_vinculo_' + papel] = status
        rel.update({'vinculos/' + papel + '/' + str(k): int(v) for k, v in status.value_counts().items()})
        mascara = status.ne('VINCULADO_COM_POPULACAO')
        for (cod, st), q in pd.DataFrame({'codigo': codigo.fillna('<AUSENTE>'), 'status': status})[mascara].value_counts().items():
            pendencias[f'{ano}|{papel}|{cod}|{st}'] += int(q)
    return out


def sql(schema):
    def tipo(t):
        if pa.types.is_string(t): return 'TEXT'
        if pa.types.is_boolean(t): return 'BOOLEAN'
        if pa.types.is_date32(t): return 'DATE'
        if pa.types.is_decimal(t): return f'NUMERIC({t.precision},{t.scale})'
        return 'BIGINT'
    cols = [f'    "{f.name}" {tipo(f.type)}' for f in schema]
    cols.append('    PRIMARY KEY ("arquivo_origem", "linha_arquivo")')
    return '-- Nao executado pelo Python. Revisar qualidade antes de carregar.\nCREATE SCHEMA IF NOT EXISTS silver;\nCREATE TABLE silver.sih (\n' + ',\n'.join(cols) + '\n);\n'


def main():
    for p in [DADOS, QUALIDADE, DOC]: p.mkdir(parents=True, exist_ok=True)
    lock = QUALIDADE / 'execucao.lock'
    try:
        f = lock.open('x')
    except FileExistsError:
        raise RuntimeError(f'Outra execucao ou interrupcao anterior: {lock}. Confirme que nao ha script rodando antes de remover SOMENTE esse lock.')
    with f: f.write(datetime.now(timezone.utc).isoformat())
    try:
        return executar()
    finally:
        lock.unlink(missing_ok=True)


def executar():
    print('SIH Silver: conferindo entradas e referencia IBGE...', flush=True)
    ref = referencia()
    arquivos = sorted(BRONZE.rglob('*.parquet'))
    inventario, combinacoes, anual, originais = {}, set(), Counter(), None
    for p in arquivos:
        m = re.fullmatch(r'RD([A-Z]{2})([0-9]{2})([0-9]{2})', p.stem)
        if not m: raise ValueError(f'Arquivo fora do padrao: {p}')
        uf, aa, mm = m.groups()
        ano, mes = 2000 + int(aa), int(mm)
        chave = (uf, ano, mes)
        if chave in combinacoes: raise ValueError(f'Competencia repetida: {chave}')
        combinacoes.add(chave)
        with pq.ParquetFile(p) as pf:
            cols = pf.schema_arrow.names
            if not OBRIGATORIAS <= set(cols): raise ValueError(f'Colunas faltando: {p}')
            if not all(pa.types.is_string(f.type) or pa.types.is_large_string(f.type) for f in pf.schema_arrow):
                raise ValueError(f'Tipos diferentes de texto: {p}')
            if originais is None: originais = cols
            if cols != originais: raise ValueError(f'Layout diferente: {p}')
            linhas = pf.metadata.num_rows
        anual[ano] += linhas
        stat = p.stat()
        inventario[p.name] = {'caminho': str(p.resolve()), 'tamanho': stat.st_size, 'mtime_ns': stat.st_mtime_ns, 'linhas': linhas}
    esperado = {(u, a, m) for u in UFS for a in range(2020, 2025) for m in range(1, 13)}
    if combinacoes != esperado: raise ValueError(f'Cobertura diferente: faltam {esperado-combinacoes}; extras {combinacoes-esperado}')
    if dict(anual) != ESPERADO: raise ValueError(f'Contagens mudaram: {dict(anual)}. Revisar antes de alterar ESPERADO.')
    schema = esquema(originais)
    config = {'versao': VERSAO, 'ibge_sha256': sha256(IBGE), 'entradas': inventario, 'schema': str(schema)}
    # Mudancas nas regras exigem nova EXECUCAO; hash do script protege retomada.
    config['versao_tratamento'] = VERSAO
    manifest = QUALIDADE / 'manifesto_tratamento.json'
    if manifest.exists():
        estado = json.loads(manifest.read_text(encoding='utf-8'))
        if estado['config'] != config: raise ValueError('Entradas, referencia ou script mudaram. Use outro EXECUCAO para nao misturar resultados.')
    else:
        estado = {'config': config, 'concluidos': {}, 'status': 'EM_EXECUCAO'}
        salvar_json(manifest, estado)
    (DOC / 'criar_tabela_sih.sql').write_text(sql(schema), encoding='utf-8')
    (DOC / 'regras.txt').write_text('Todos os originais preservados como texto; nenhuma deduplicacao ou filtro clinico.\nChave tecnica: arquivo_origem + linha_arquivo (1-based). N_AIH nao e chave unica.\nVinculo IBGE exato pelos seis primeiros digitos do codigo oficial, usando ano_processamento.\nMunicipios sem match ficam nulos; sem equivalencias automaticas.\nDatas YYYYMMDD entre 1800-01-01 e 2100-12-31; originais preservados.\nValores VAL_* em decimal(18,2), sem arredondar entradas com mais de duas casas.\nIDADE_numerico mantem unidade indicada por COD_IDADE; nao representa necessariamente anos.\nUS_TOT permanece texto. Nenhuma conversao cambial.\nFlags cronologicas false com datas ausentes: consultar tambem status das datas.\nSem vinculo direto a CNES leitos: essa base nao contem necessariamente todos os estabelecimentos SIH.\nRelatorio exige revisao, mesmo com conversao concluida.\nFonte semantica: https://wiki.saude.gov.br/sih/index.php/P%C3%A1gina_principal\n', encoding='utf-8')
    try:
        for i, p in enumerate(arquivos, 1):
            uf, aa, mm = re.fullmatch(r'RD([A-Z]{2})([0-9]{2})([0-9]{2})', p.stem).groups()
            ano, mes = 2000 + int(aa), int(mm)
            destino = DADOS / f'uf={uf}' / f'competencia={ano}{mes:02d}' / p.name
            destino.parent.mkdir(parents=True, exist_ok=True)
            anterior = estado['concluidos'].get(p.name)
            if anterior:
                if not destino.is_file() or sha256(destino) != anterior['sha256_saida']:
                    raise ValueError(f'Saida concluida ausente ou alterada: {destino}')
                if sha256(p) != anterior['sha256_entrada']:
                    raise ValueError(f'Entrada alterada: {p}')
                if i % 100 == 0: print(f'Reutilizados/verificados: {i}/{len(arquivos)}', flush=True)
                continue
            print(f'[{i}/{len(arquivos)}] Tratando {p.name}', flush=True)
            digest = sha256(p)
            parcial = destino.with_suffix('.parquet.part')
            contagem, rel, pend = 0, Counter(), Counter()
            instante = datetime.now(timezone.utc).isoformat()
            with pq.ParquetFile(p) as pf, pq.ParquetWriter(parcial, schema, compression='snappy') as writer:
                for batch in pf.iter_batches(batch_size=LOTE):
                    df = batch.to_pandas().reset_index(drop=True)
                    out = tratar(df, ano, mes, uf, p.name, contagem, ref, rel, pend, instante)
                    tab = pa.Table.from_pandas(out, schema=schema, preserve_index=False, safe=True)
                    writer.write_table(tab)
                    contagem += len(df)
            with pq.ParquetFile(parcial) as ver:
                if ver.metadata.num_rows != inventario[p.name]['linhas'] or contagem != ver.metadata.num_rows:
                    raise ValueError(f'Contagem de saida incorreta: {p}')
            if sha256(p) != digest: raise ValueError(f'Entrada mudou durante leitura: {p}')
            parcial.replace(destino)
            estado['concluidos'][p.name] = {'registros': contagem, 'ano': ano, 'saida': str(destino), 'sha256_entrada': digest, 'sha256_saida': sha256(destino), 'contagens': dict(rel), 'pendencias': dict(pend)}
            salvar_json(manifest, estado)
        total, pend, anual_saida = Counter(), Counter(), Counter()
        for r in estado['concluidos'].values():
            total.update(r['contagens']); pend.update(r['pendencias']); anual_saida[r['ano']] += r['registros']
        if dict(anual_saida) != ESPERADO: raise ValueError('Contagens finais divergentes.')
        resumo = {'status': 'REVISAR_ANTES_DA_CARGA', 'registros': sum(anual_saida.values()), 'arquivos': len(estado['concluidos']), 'registros_por_ano_processamento': dict(anual_saida), 'contagens': dict(sorted(total.items())), 'dados': str(DADOS), 'observacao': 'Linhas preservadas; nao sao contagem de internacoes distintas. Nenhuma carga PostgreSQL.'}
        salvar_json(QUALIDADE / 'relatorio_qualidade.json', resumo)
        registros = [dict(zip(['ano_processamento', 'papel', 'codigo', 'status'], k.split('|')), registros=v) for k, v in sorted(pend.items())]
        pd.DataFrame(registros, columns=['ano_processamento', 'papel', 'codigo', 'status', 'registros']).to_csv(QUALIDADE / 'pendencias_municipais.csv', index=False, sep=';', encoding='utf-8-sig')
        estado['status'] = resumo['status']; estado.pop('erro', None)
        salvar_json(manifest, estado)
        print(f"\nStatus: {resumo['status']}\nRegistros: {resumo['registros']:,}\nDados: {DADOS}\nQualidade: {QUALIDADE}\nSQL: {DOC / 'criar_tabela_sih.sql'}", flush=True)
        print('Nenhuma carga foi realizada no PostgreSQL.', flush=True)
        return resumo
    except Exception:
        estado['status'] = 'INTERROMPIDO'
        estado['erro'] = traceback.format_exc()
        salvar_json(manifest, estado)
        raise


if __name__ == '__main__':
    resultado_sih = main()


SIH Silver: conferindo entradas e referencia IBGE...
[1/1620] Tratando RDAC2001.parquet
[2/1620] Tratando RDAC2002.parquet
[3/1620] Tratando RDAC2003.parquet
[4/1620] Tratando RDAC2004.parquet
[5/1620] Tratando RDAC2005.parquet
[6/1620] Tratando RDAC2006.parquet
[7/1620] Tratando RDAC2007.parquet
[8/1620] Tratando RDAC2008.parquet
[9/1620] Tratando RDAC2009.parquet
[10/1620] Tratando RDAC2010.parquet
[11/1620] Tratando RDAC2011.parquet
[12/1620] Tratando RDAC2012.parquet
[13/1620] Tratando RDAC2101.parquet
[14/1620] Tratando RDAC2102.parquet
[15/1620] Tratando RDAC2103.parquet
[16/1620] Tratando RDAC2104.parquet
[17/1620] Tratando RDAC2105.parquet
[18/1620] Tratando RDAC2106.parquet
[19/1620] Tratando RDAC2107.parquet
[20/1620] Tratando RDAC2108.parquet
[21/1620] Tratando RDAC2109.parquet
[22/1620] Tratando RDAC2110.parquet
[23/1620] Tratando RDAC2111.parquet
[24/1620] Tratando RDAC2112.parquet
[25/1620] Tratando RDAC2201.parquet
[26/1620] Tratando RDAC2202.parquet
[27/1620] Tratando R

In [7]:
from pathlib import Path
import json

pasta = Path(
    r"D:\Cleber\GitHub\Projetos\observatorio_covid19_brasil"
    r"\Camada_Silver\qualidade\sih\sih_2020_2024_v1"
)

relatorio = json.loads(
    (pasta / "relatorio_qualidade.json").read_text(encoding="utf-8")
)

print(json.dumps(relatorio, ensure_ascii=False, indent=2))

{
  "status": "REVISAR_ANTES_DA_CARGA",
  "registros": 62365300,
  "arquivos": 1620,
  "registros_por_ano_processamento": {
    "2020": 10688203,
    "2021": 11629005,
    "2022": 12520914,
    "2023": 13355814,
    "2024": 14171364
  },
  "contagens": {
    "avisos/fl_dois_anos_internacao_processamento_diferentes": 4052756,
    "avisos/fl_internacao_fora_2020_2024": 926011,
    "avisos/fl_nascimento_apos_internacao": 0,
    "avisos/fl_saida_antes_internacao": 0,
    "categorias/COD_IDADE/0": 12,
    "categorias/COD_IDADE/2": 1969491,
    "categorias/COD_IDADE/3": 1182084,
    "categorias/COD_IDADE/4": 59166829,
    "categorias/COD_IDADE/5": 46884,
    "categorias/IDENT/1": 61645193,
    "categorias/IDENT/5": 720107,
    "categorias/MORTE/0": 59187699,
    "categorias/MORTE/1": 3177601,
    "categorias/SEXO/1": 26758668,
    "categorias/SEXO/3": 35606632,
    "categorias/_DBF_EXCLUIDO/0": 62365300,
    "datas/data_internacao/VALIDA": 62365300,
    "datas/data_nascimento/VALIDA": 623653

## PostgreSQL

In [1]:
from pathlib import Path
from getpass import getpass
import hashlib
import json

import pyarrow.parquet as pq
import psycopg
from psycopg import sql


PASTA_QUALIDADE = Path(
    r"D:\Cleber\GitHub\Projetos\observatorio_covid19_brasil"
    r"\Camada_Silver\qualidade\sih\sih_2020_2024_v1"
)

CONEXAO = {
    "host": "localhost",
    "port": 5432,
    "dbname": "observatorio_covid19_brasil",
    "user": "postgres",
}

LOTE = 5_000


def sha256_arquivo(caminho):
    h = hashlib.sha256()
    with caminho.open("rb") as arquivo:
        for bloco in iter(lambda: arquivo.read(8 * 1024 * 1024), b""):
            h.update(bloco)
    return h.hexdigest()


def carregar_sih():
    manifesto = json.loads(
        (PASTA_QUALIDADE / "manifesto_tratamento.json")
        .read_text(encoding="utf-8")
    )
    relatorio = json.loads(
        (PASTA_QUALIDADE / "relatorio_qualidade.json")
        .read_text(encoding="utf-8")
    )

    concluidos = manifesto["concluidos"]
    esperado = int(relatorio["registros"])

    if len(concluidos) != 1620:
        raise RuntimeError("O manifesto não contém os 1.620 arquivos concluídos.")

    if sum(int(r["registros"]) for r in concluidos.values()) != esperado:
        raise RuntimeError("Contagens do manifesto e relatório divergem.")

    arquivos = []
    for nome, registro in sorted(concluidos.items()):
        caminho = Path(registro["saida"])
        if not caminho.is_file():
            raise FileNotFoundError(caminho)
        arquivos.append((nome, caminho, registro))

    senha = getpass("Senha do PostgreSQL: ")

    # autocommit permite controlar uma transação por arquivo.
    with psycopg.connect(**CONEXAO, password=senha, autocommit=True) as conn:
        with conn.cursor() as cur:
            # Impede duas instâncias deste carregador simultâneas.
            cur.execute("SELECT pg_try_advisory_lock(20261029, 62)")
            if not cur.fetchone()[0]:
                raise RuntimeError("Já existe outra carga SIH em execução.")

            try:
                cur.execute("SELECT to_regclass('silver.sih')")
                if cur.fetchone()[0] is None:
                    raise RuntimeError("A tabela silver.sih não existe neste banco.")

                cur.execute("""
                    CREATE TABLE IF NOT EXISTS silver.controle_carga_sih (
                        arquivo_origem TEXT PRIMARY KEY,
                        sha256_parquet TEXT NOT NULL,
                        registros BIGINT NOT NULL,
                        carregado_em TIMESTAMPTZ NOT NULL DEFAULT now()
                    )
                """)

                cur.execute("""
                    SELECT arquivo_origem, sha256_parquet, registros
                    FROM silver.controle_carga_sih
                """)
                controle = {
                    nome: (digest, quantidade)
                    for nome, digest, quantidade in cur.fetchall()
                }

                if set(controle) - set(concluidos):
                    raise RuntimeError(
                        "O banco contém arquivos de outra coleta. "
                        "A carga foi interrompida para revisão."
                    )

                # Verifica o estado inicial, inclusive em uma retomada.
                cur.execute("SELECT COUNT(*) FROM silver.sih")
                total_banco = cur.fetchone()[0]
                total_controlado = sum(q for _, q in controle.values())

                if total_banco != total_controlado:
                    raise RuntimeError(
                        f"Tabela com {total_banco:,} linhas, mas controle com "
                        f"{total_controlado:,}. Revisar antes de continuar."
                    )

                cur.execute("""
                    SELECT column_name
                    FROM information_schema.columns
                    WHERE table_schema = 'silver'
                      AND table_name = 'sih'
                    ORDER BY ordinal_position
                """)
                colunas_banco = [r[0] for r in cur.fetchall()]

                print(
                    f"Total esperado: {esperado:,}\n"
                    f"Já carregados: {total_controlado:,}",
                    flush=True,
                )

                for indice, (nome, caminho, registro) in enumerate(arquivos, 1):
                    digest = sha256_arquivo(caminho)

                    if digest != registro["sha256_saida"]:
                        raise RuntimeError(f"Parquet alterado: {caminho}")

                    quantidade = int(registro["registros"])

                    if nome in controle:
                        if controle[nome] != (digest, quantidade):
                            raise RuntimeError(f"Controle divergente: {nome}")
                        if indice % 100 == 0:
                            print(
                                f"Verificados/reutilizados: {indice}/1620",
                                flush=True,
                            )
                        continue

                    with pq.ParquetFile(caminho) as parquet:
                        colunas = parquet.schema_arrow.names

                        if colunas != colunas_banco:
                            raise RuntimeError(
                                f"Colunas do Parquet e da tabela divergem: {nome}"
                            )

                        if parquet.metadata.num_rows != quantidade:
                            raise RuntimeError(f"Contagem divergente: {nome}")

                        comando = sql.SQL(
                            "COPY silver.sih ({}) FROM STDIN"
                        ).format(
                            sql.SQL(", ").join(map(sql.Identifier, colunas))
                        )

                        # Dados e controle são confirmados juntos.
                        # Em caso de erro, este arquivo inteiro é desfeito.
                        with conn.transaction():
                            enviados = 0

                            with cur.copy(comando) as copia:
                                for lote in parquet.iter_batches(batch_size=LOTE):
                                    valores = [
                                        coluna.to_pylist()
                                        for coluna in lote.columns
                                    ]
                                    for linha in zip(*valores):
                                        copia.write_row(linha)
                                    enviados += lote.num_rows

                            if enviados != quantidade:
                                raise RuntimeError(
                                    f"Quantidade enviada incorreta: {nome}"
                                )

                            cur.execute("""
                                SELECT COUNT(*)
                                FROM silver.sih
                                WHERE arquivo_origem = %s
                            """, (nome,))

                            if cur.fetchone()[0] != quantidade:
                                raise RuntimeError(
                                    f"Contagem no banco divergente: {nome}"
                                )

                            cur.execute("""
                                INSERT INTO silver.controle_carga_sih
                                    (arquivo_origem, sha256_parquet, registros)
                                VALUES (%s, %s, %s)
                            """, (nome, digest, quantidade))

                    total_controlado += quantidade
                    print(
                        f"[{indice}/1620] {nome} confirmado | "
                        f"Total: {total_controlado:,} / {esperado:,}",
                        flush=True,
                    )

                print("Conferindo totais por ano de processamento...", flush=True)

                cur.execute("""
                    SELECT ano_processamento, COUNT(*)
                    FROM silver.sih
                    GROUP BY ano_processamento
                    ORDER BY ano_processamento
                """)
                por_ano = dict(cur.fetchall())
                esperado_por_ano = {
                    int(ano): int(q)
                    for ano, q in
                    relatorio["registros_por_ano_processamento"].items()
                }

                if por_ano != esperado_por_ano:
                    raise RuntimeError(
                        f"Contagens anuais divergentes: {por_ano}. "
                        "Os arquivos já confirmados permanecem no banco."
                    )

                print("\nCarga concluída!", flush=True)
                print(f"Registros no banco: {sum(por_ano.values()):,}")
                print("Contagens anuais conferidas.")
                print("Ressalvas e valores originais preservados.")

            finally:
                cur.execute("SELECT pg_advisory_unlock(20261029, 62)")


carregar_sih()

Total esperado: 62,365,300
Já carregados: 0
[1/1620] RDAC2001.parquet confirmado | Total: 3,784 / 62,365,300
[2/1620] RDAC2002.parquet confirmado | Total: 7,873 / 62,365,300
[3/1620] RDAC2003.parquet confirmado | Total: 11,644 / 62,365,300
[4/1620] RDAC2004.parquet confirmado | Total: 14,606 / 62,365,300
[5/1620] RDAC2005.parquet confirmado | Total: 17,718 / 62,365,300
[6/1620] RDAC2006.parquet confirmado | Total: 20,909 / 62,365,300
[7/1620] RDAC2007.parquet confirmado | Total: 24,397 / 62,365,300
[8/1620] RDAC2008.parquet confirmado | Total: 28,173 / 62,365,300
[9/1620] RDAC2009.parquet confirmado | Total: 31,671 / 62,365,300
[10/1620] RDAC2010.parquet confirmado | Total: 35,917 / 62,365,300
[11/1620] RDAC2011.parquet confirmado | Total: 39,529 / 62,365,300
[12/1620] RDAC2012.parquet confirmado | Total: 43,227 / 62,365,300
[13/1620] RDAC2101.parquet confirmado | Total: 47,160 / 62,365,300
[14/1620] RDAC2102.parquet confirmado | Total: 50,654 / 62,365,300
[15/1620] RDAC2103.parquet co